# Hierarchical Muay Thai Strike Classification Agent

This notebook tests the hierarchical agent (coarse → fine classification) with video files and saves the results.


In [5]:
# Import necessary libraries
import json
import os
from datetime import datetime
from pathlib import Path

# Import the hierarchical agent graph
from hierarchical_agent import hierarchical_graph, llm
from langchain_core.messages import HumanMessage


In [6]:
# Load true labels for comparison
with open('data/combos/true_labels.json', 'r') as f:
    true_labels = json.load(f)

print("True labels loaded:")
for video_id, strikes in list(true_labels.items())[:5]:
    print(f"  Video {video_id}: {strikes}")


True labels loaded:
  Video 1: ['jab', 'cross', 'roundhouse']
  Video 2: ['jab', 'jab', 'cross', 'roundhouse']
  Video 3: ['jab', 'cross', 'lead_hook', 'lead_hook']
  Video 4: ['teep', 'roundhouse']
  Video 5: ['lead_uppercut', 'cross', 'lead_hook']


In [7]:
# Configuration
VIDEO_DIR = "data/combos"
RESULTS_DIR = "results_hierarchical"

# Create results directory if it doesn't exist
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_result(video_id, video_path, predicted_strikes, coarse_categories=None, true_strikes=None, full_response=None):
    """Save the classification result to a JSON file."""
    result = {
        "video_id": video_id,
        "video_path": video_path,
        "timestamp": datetime.now().isoformat(),
        "coarse_categories": coarse_categories,
        "predicted_strikes": predicted_strikes,
        "true_strikes": true_strikes,
        "correct": predicted_strikes == true_strikes if true_strikes else None,
        "full_response": full_response
    }
    
    filename = f"{RESULTS_DIR}/result_{video_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent=2)
    
    print(f"\nResult saved to: {filename}")
    return result


## Test Single Video


In [8]:
# Test a single video with hierarchical agent
video_id = "1"
video_path = f"{VIDEO_DIR}/{video_id}.mov"

print(f"Hierarchical agent analyzing video: {video_path}")
print("=" * 60)

# Get true labels if available
true_strikes = true_labels.get(video_id)
if true_strikes:
    print(f"True strikes: {true_strikes}")
    print("-" * 60)

# Invoke the hierarchical graph
result = hierarchical_graph.invoke(
    {
        "messages": [HumanMessage(content=f"Classify the strikes in this video: {video_path}")],
        "video_path": video_path
    }
)

# Extract the response (handle both string and list formats)
agent_response_raw = result["messages"][-1].content
print(f"\nAgent Response (raw): {agent_response_raw}")

# Extract text from content (could be string or list of content blocks)
if isinstance(agent_response_raw, list):
    agent_response = ""
    for block in agent_response_raw:
        if isinstance(block, dict) and 'text' in block:
            agent_response += block['text']
        elif isinstance(block, str):
            agent_response += block
    print(f"Extracted text: {agent_response}")
else:
    agent_response = str(agent_response_raw)

print(f"\nAgent Response:")
print(agent_response)

# Extract coarse categories and fine strikes from response
coarse_categories = result.get("coarse_categories", [])
fine_strikes = result.get("fine_strikes", [])

# Try to parse JSON from response - look for "Final classification:" line first
import re
predicted_strikes = None

# First try to get from state (most reliable)
if fine_strikes:
    predicted_strikes = fine_strikes
    print(f"\nStrikes from state: {predicted_strikes}")
else:
    # Try to extract from "Final classification:" line
    final_match = re.search(r'Final classification:\s*(\[.*?\])', agent_response)
    if final_match:
        try:
            predicted_strikes = json.loads(final_match.group(1))
            print(f"\nParsed from 'Final classification': {predicted_strikes}")
        except json.JSONDecodeError:
            pass
    
    # Fallback: try to find any JSON array
    if not predicted_strikes:
        json_match = re.search(r'\[.*?\]', agent_response)
        if json_match:
            try:
                predicted_strikes = json.loads(json_match.group())
                print(f"\nParsed strikes from JSON array: {predicted_strikes}")
            except json.JSONDecodeError:
                pass

if predicted_strikes:
    print(f"\nParsed strikes: {predicted_strikes}")
    
    # Compare with true labels
    if true_strikes:
        print(f"\nComparison:")
        print(f"  True:     {true_strikes}")
        print(f"  Predicted: {predicted_strikes}")
        print(f"  Match:    {predicted_strikes == true_strikes}")
        if coarse_categories:
            print(f"  Coarse categories: {coarse_categories}")
    
    # Save result
    save_result(video_id, video_path, predicted_strikes, coarse_categories, true_strikes, agent_response)
else:
    print("\nCould not parse strikes from response")
    print(f"Full response: {agent_response}")
    save_result(video_id, video_path, None, coarse_categories, true_strikes, agent_response)


Hierarchical agent analyzing video: data/combos/1.mov
True strikes: ['jab', 'cross', 'roundhouse']
------------------------------------------------------------

[COARSE] Classifying strike categories...
[DEBUG] analyze_video (coarse) returned: ["kick"]...
[COARSE] Response (raw): [{'type': 'text', 'text': '```json\n["kick"]\n```', 'extras': {'signature': 'CpcEAdHtim8pJzxi0FY4zGyuHm72QqYqMHVIex3cViNWlG3MHnejTgjl51Xfi6h1QfcNmpE79vccnjPpTdaElQEpU+Lg1BFXdG0vOeuXLe2fBgx/aExqEwt83DmkEchepncBwhhsUshV1PKYxp6mK1euNUEepL9z4ndscQFV1IUokimYm6BnvW7Fa6txPgZmws+b9yzgl4cDJfaH02OmE+s3Bt7B+5tg7wSuMW4rFgHglAUZ0G04buDE0YnvJujBRIzzzIrchIkmptnQGNT4PnPOGUKbWJl+ihVXE+GOoGoj4DdDid6cz3RlSnFSbCzPYBat/bxTob63OjGpDzSYYX7zMcWzFXpduif1zUNIPzahqYdcK42r9pQCvwt/62cJqfFtHLiU3ojRpfKdiNtyWXgT+wrJ6m8xzr0orplXvKGMLqyxpKkpWx64LileCAJv+bxfdtShMgaYk3YFQ8EPMFyy/jF4GV/SNsfWCt3wVY0+ybnfwgE03H6DtoX6NI7x+eU6UpXyP76fDHoxmr3TJgbby067vvuKtVYQNQbzpViTdjnjyWoAR/ONbkUF+HRCsZo30yAHMJhRUc1CEsDJIVLW1CTtR28fm0drvwY/Q8jPv/1NQCaPR360oW3O2AY8t6

## Test Multiple Videos

In [13]:
from time import sleep

# Test multiple videos with hierarchical agent
video_ids = ["1", "2", "3", "4", "5"]  # Adjust as needed

all_results = []

for video_id in video_ids:
    video_path = f"{VIDEO_DIR}/{video_id}.mov"
    
    if not os.path.exists(video_path):
        print(f"Skipping {video_id}: file not found")
        continue
    
    print(f"\n{'='*60}")
    print(f"Analyzing video {video_id}: {video_path}")
    print(f"{'='*60}")
    
    true_strikes = true_labels.get(video_id)
    if true_strikes:
        print(f"True strikes: {true_strikes}")
    
    try:
        # Invoke the hierarchical graph
        result = hierarchical_graph.invoke(
            {
                "messages": [HumanMessage(content=f"Classify the strikes in this video: {video_path}")],
                "video_path": video_path
            }
        )
        
        agent_response_raw = result["messages"][-1].content
        
        # Extract text from content (could be string or list of content blocks)
        if isinstance(agent_response_raw, list):
            agent_response = ""
            for block in agent_response_raw:
                if isinstance(block, dict) and 'text' in block:
                    agent_response += block['text']
                elif isinstance(block, str):
                    agent_response += block
        else:
            agent_response = str(agent_response_raw)
        
        coarse_categories = result.get("coarse_categories", [])
        fine_strikes = result.get("fine_strikes", [])
        print(f"\nResponse: {agent_response[:200]}...")
        if coarse_categories:
            print(f"Coarse categories: {coarse_categories}")
        if fine_strikes:
            print(f"Fine strikes from state: {fine_strikes}")
        
        # Try to parse JSON - prefer state, then "Final classification:" line
        import re
        predicted_strikes = None
        
        # First try state
        if fine_strikes:
            predicted_strikes = fine_strikes
            print(f"Using strikes from state: {predicted_strikes}")
        else:
            # Try to extract from "Final classification:" line
            final_match = re.search(r'Final classification:\s*(\[.*?\])', agent_response)
            if final_match:
                try:
                    predicted_strikes = json.loads(final_match.group(1))
                    print(f"Parsed from 'Final classification': {predicted_strikes}")
                except json.JSONDecodeError:
                    pass
            
            # Fallback: try to find any JSON array
            if not predicted_strikes:
                json_match = re.search(r'\[.*?\]', agent_response)
                if json_match:
                    try:
                        predicted_strikes = json.loads(json_match.group())
                        print(f"Parsed from JSON array: {predicted_strikes}")
                    except json.JSONDecodeError:
                        print("Could not parse JSON")
        
        if predicted_strikes and true_strikes:
            match = predicted_strikes == true_strikes
            print(f"Match: {match}")
        
        # Save result
        result_data = save_result(video_id, video_path, predicted_strikes, coarse_categories, true_strikes, agent_response)
        all_results.append(result_data)
        
    except Exception as e:
        print(f"Error processing video {video_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        save_result(video_id, video_path, None, None, true_strikes, f"Error: {str(e)}")

    sleep(60)

print(f"\n\nCompleted {len(all_results)} videos")



Analyzing video 1: data/combos/1.mov
True strikes: ['jab', 'cross', 'roundhouse']

[COARSE] Classifying strike categories...
[DEBUG] analyze_video (coarse) returned: ["kick"]...
[COARSE] Response (raw): [{'type': 'text', 'text': '```json\n["kick"]\n```', 'extras': {'signature': 'CqoEAdHtim/ZB6qrRhJ0MxyiXVP8WrQRHHSbAUq5HOst9+cmcOth1XCYiYM1bPKZKcHl8EUfg9kmKm76LFV4/08pf3Muz5LwEkm+DLN2qQOtcbgVeLY5A/2F9JnS0gjKYna5H5cbCZczyJSUPQK5wHuwdVKmRT+m6TsjgWTHz+RijqFZ8wXvA+k2pRQ0bUiGiOy0mFkokCa2heg5d2D4k9C9Djh6gviK41rXH5NsaW6nggCpyusvXJHgSda7oHqgmxIubTULm8k8n2rsrkWDkpnPlpaHQtICJNbaLrnTkM+IU9rli2kbm1qYgZQTAuvLJrZTd5d3LUHZvRbZGUgg3mcGQg7/0whyLQM/Kn/A/mBkDsoOHNk9lmEspBMG/h5L23vQRt4Rvh3kpcwMysix4wfI/lXZ0jRH50e+3FSmXy1eNuehBSn7fl+qZfh2FqyOhJ3bGIu9KQEGOyaY8Gl9B6jn4ulHWGVLzQp83pkUXZDyVSVC1VPNM8z9OsGDz7m1lhx8nZNtcc2+3sEoj37mAFd/grHaGobxLSK0gYHK96CuwUTPyf/IsLYDeUeoGQ9wMyqPIo2eoQELBZNzetRFch9PRYXzZ3mG3+0m4gvYoA/LWxVZhg6U9kYG1Qja+WfuVd8FRoUASZZwE4guIM/hhlpte7hSwGS5XDBi9YPb3ewmbIri4YTMtD+oJtMXWe/PJETbZLSGt2uOyPe

## Analyze Results


In [14]:
# Load and analyze saved results
import glob

result_files = glob.glob(f"{RESULTS_DIR}/result_*.json")

if result_files:
    # Get the most recent results
    result_files.sort(key=os.path.getmtime, reverse=True)
    
    print(f"Found {len(result_files)} result files")
    print("\nMost recent results:")
    
    for filepath in result_files[:10]:  # Show last 10
        with open(filepath, 'r') as f:
            result = json.load(f)
        
        video_id = result.get('video_id', 'unknown')
        predicted = result.get('predicted_strikes', 'N/A')
        true = result.get('true_strikes', 'N/A')
        coarse = result.get('coarse_categories', 'N/A')
        correct = result.get('correct', 'N/A')
        
        print(f"\nVideo {video_id}:")
        print(f"  True:     {true}")
        print(f"  Predicted: {predicted}")
        print(f"  Coarse:   {coarse}")
        print(f"  Correct:  {correct}")
else:
    print("No result files found")


Found 18 result files

Most recent results:

Video 5:
  True:     ['lead_uppercut', 'cross', 'lead_hook']
  Predicted: ['punch', 'punch']
  Coarse:   ['punch', 'punch']
  Correct:  False

Video 4:
  True:     ['teep', 'roundhouse']
  Predicted: None
  Coarse:   []
  Correct:  False

Video 3:
  True:     ['jab', 'cross', 'lead_hook', 'lead_hook']
  Predicted: ['jab', 'cross', 'kick']
  Coarse:   ['punch', 'punch', 'kick']
  Correct:  False

Video 2:
  True:     ['jab', 'jab', 'cross', 'roundhouse']
  Predicted: ['jab', 'cross']
  Coarse:   ['punch', 'punch']
  Correct:  False

Video 1:
  True:     ['jab', 'cross', 'roundhouse']
  Predicted: ['roundhouse']
  Coarse:   ['kick']
  Correct:  False

Video 5:
  True:     ['lead_uppercut', 'cross', 'lead_hook']
  Predicted: None
  Coarse:   None
  Correct:  False

Video 4:
  True:     ['teep', 'roundhouse']
  Predicted: ['knee']
  Coarse:   ['knee']
  Correct:  False

Video 3:
  True:     ['jab', 'cross', 'lead_hook', 'lead_hook']
  Predicted:

In [15]:
# Calculate accuracy
if result_files:
    results_with_labels = []
    
    for filepath in result_files:
        with open(filepath, 'r') as f:
            result = json.load(f)
        
        if result.get('true_strikes') and result.get('predicted_strikes'):
            results_with_labels.append(result)
    
    if results_with_labels:
        correct = sum(1 for r in results_with_labels if r.get('correct') == True)
        total = len(results_with_labels)
        accuracy = (correct / total) * 100 if total > 0 else 0
        
        print(f"\nAccuracy Summary:")
        print(f"  Correct: {correct}/{total}")
        print(f"  Accuracy: {accuracy:.1f}%")
        
        # Show breakdown by coarse category
        print(f"\nCoarse Category Breakdown:")
        category_counts = {}
        for r in results_with_labels:
            coarse = r.get('coarse_categories', [])
            for cat in coarse:
                category_counts[cat] = category_counts.get(cat, 0) + 1
        
        for cat, count in sorted(category_counts.items()):
            print(f"  {cat}: {count}")



Accuracy Summary:
  Correct: 0/9
  Accuracy: 0.0%

Coarse Category Breakdown:
  kick: 6
  knee: 1
  punch: 6


## Compare Hierarchical vs Basic Agent

Compare results between hierarchical and basic agent approaches.


In [16]:
# Compare hierarchical results with basic agent results
import glob as glob_module

hierarchical_files = glob_module.glob(f"{RESULTS_DIR}/result_*.json")
basic_files = glob_module.glob("results/result_*.json")

if hierarchical_files and basic_files:
    print("Comparing Hierarchical vs Basic Agent Results:")
    print("=" * 60)
    
    # Get most recent result for each video from each approach
    hierarchical_by_video = {}
    basic_by_video = {}
    
    for filepath in hierarchical_files:
        with open(filepath, 'r') as f:
            result = json.load(f)
        video_id = result.get('video_id')
        if video_id and (video_id not in hierarchical_by_video or 
                        os.path.getmtime(filepath) > os.path.getmtime(hierarchical_by_video[video_id])):
            hierarchical_by_video[video_id] = filepath
    
    for filepath in basic_files:
        with open(filepath, 'r') as f:
            result = json.load(f)
        video_id = result.get('video_id')
        if video_id and (video_id not in basic_by_video or 
                        os.path.getmtime(filepath) > os.path.getmtime(basic_by_video[video_id])):
            basic_by_video[video_id] = filepath
    
    # Compare common videos
    common_videos = set(hierarchical_by_video.keys()) & set(basic_by_video.keys())
    
    if common_videos:
        print(f"\nFound {len(common_videos)} videos in both approaches:")
        for video_id in sorted(common_videos):
            with open(hierarchical_by_video[video_id], 'r') as f:
                h_result = json.load(f)
            with open(basic_by_video[video_id], 'r') as f:
                b_result = json.load(f)
            
            print(f"\nVideo {video_id}:")
            print(f"  True:           {h_result.get('true_strikes', 'N/A')}")
            print(f"  Hierarchical:   {h_result.get('predicted_strikes', 'N/A')} (Coarse: {h_result.get('coarse_categories', 'N/A')})")
            print(f"  Basic:          {b_result.get('predicted_strikes', 'N/A')}")
            print(f"  Hierarchical ✓: {h_result.get('correct', 'N/A')}")
            print(f"  Basic ✓:        {b_result.get('correct', 'N/A')}")
    else:
        print("No common videos found for comparison")
else:
    print("Need results from both approaches to compare")


Comparing Hierarchical vs Basic Agent Results:

Found 5 videos in both approaches:

Video 1:
  True:           ['jab', 'cross', 'roundhouse']
  Hierarchical:   ['roundhouse'] (Coarse: ['kick'])
  Basic:          ['teep']
  Hierarchical ✓: False
  Basic ✓:        False

Video 2:
  True:           ['jab', 'jab', 'cross', 'roundhouse']
  Hierarchical:   ['jab', 'cross'] (Coarse: ['punch', 'punch'])
  Basic:          None
  Hierarchical ✓: False
  Basic ✓:        False

Video 3:
  True:           ['jab', 'cross', 'lead_hook', 'lead_hook']
  Hierarchical:   ['jab', 'cross', 'kick'] (Coarse: ['punch', 'punch', 'kick'])
  Basic:          None
  Hierarchical ✓: False
  Basic ✓:        False

Video 4:
  True:           ['teep', 'roundhouse']
  Hierarchical:   None (Coarse: [])
  Basic:          None
  Hierarchical ✓: False
  Basic ✓:        False

Video 5:
  True:           ['lead_uppercut', 'cross', 'lead_hook']
  Hierarchical:   ['punch', 'punch'] (Coarse: ['punch', 'punch'])
  Basic:        